<a href="https://colab.research.google.com/github/triman1905/email-classifier-internship-TBO-/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import spacy
import torch
import optuna

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler
from sklearn.feature_extraction.text import TfidfVectorizer

from sentence_transformers import SentenceTransformer

# 🔧 Set device
device = torch.device("cpu")

# ------------------ Load Models ------------------
print("📦 Loading models...")
nlp = spacy.load("en_core_web_sm", disable=["parser", "tagger"])
bert_model = SentenceTransformer("distiluse-base-multilingual-cased")

# ------------------ Helper Functions ------------------
def clean_text(text):
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)
    text = re.sub(r"\d{6,}", "", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.lower().strip()

def extract_ner_features(df):
    ner_labels = ["PERSON", "ORG", "GPE"]
    features = []
    for doc in nlp.pipe(df["clean_text"].tolist(), batch_size=32):
        found = [ent.label_ for ent in doc.ents]
        binary = [int(label in found) for label in ner_labels]
        features.append(binary)
    return pd.DataFrame(features, columns=[f"ent_{label}" for label in ner_labels])

def evaluate_model(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="weighted"),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0)
    }

# ------------------ Load Data ------------------
print("📥 Reading CSV...")
try:
    df = pd.read_csv("supplier_classification_sample50000.csv", quoting=0, on_bad_lines="skip", nrows=30000, encoding="utf-8")
except UnicodeDecodeError:
    print("⚠️ UTF-8 decoding failed. Trying ISO-8859-1 encoding.")
    df = pd.read_csv("supplier_classification_sample50000.csv", quoting=0, on_bad_lines="skip", nrows=30000, encoding="ISO-8859-1")

df = df[df["ticketclassification"].isin(["ACTIONABLE", "NON-ACTIONABLE"])].dropna(subset=["body"])
df["true_label"] = df["ticketclassification"].map({"ACTIONABLE": 1, "NON-ACTIONABLE": 0})
df["clean_text"] = df["body"].astype(str).map(clean_text)

# ------------------ Feature Engineering ------------------
print("🧠 Extracting NER...")
ner_df = extract_ner_features(df)

print("🔗 Generating BERT embeddings...")
bert_embeddings = bert_model.encode(df["clean_text"].tolist(), show_progress_bar=True)

print("📄 Extracting TF-IDF features...")
tfidf = TfidfVectorizer(max_features=300)
tfidf_features = tfidf.fit_transform(df["clean_text"]).toarray()

# 🔗 Combine BERT + NER + TFIDF
X = np.hstack([bert_embeddings, ner_df.values, tfidf_features])
y = df["true_label"].values

# 🧪 Balance the dataset
print("🔄 Balancing data...")
X, y = RandomOverSampler(random_state=42).fit_resample(X, y)

# ------------------ Optuna Hyperparameter Search ------------------
def objective(trial):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    hidden_sizes = [trial.suggest_int(f"layer_{i}", 50, 200) for i in range(n_layers)]
    clf = MLPClassifier(
        hidden_layer_sizes=tuple(hidden_sizes),
        activation=trial.suggest_categorical("activation", ["relu", "tanh"]),
        learning_rate_init=trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
        max_iter=300,
        random_state=42
    )
    X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", clf)
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_val)
    return f1_score(y_val, preds, average="weighted")

print("🎯 Tuning with Optuna...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)
print("✅ Best Params:", study.best_params)

# ------------------ Final Training & Evaluation ------------------
final_mlp = MLPClassifier(
    hidden_layer_sizes=tuple([study.best_params[f"layer_{i}"] for i in range(study.best_params["n_layers"])]),
    activation=study.best_params["activation"],
    learning_rate_init=study.best_params["learning_rate_init"],
    max_iter=300,
    random_state=42
)

print("🔁 Cross Validation...")
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
metrics_list = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"\n📂 Fold {fold + 1}")
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", final_mlp)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    metrics = evaluate_model(y_test, y_pred)
    metrics_list.append(metrics)
    print(metrics)

# 📊 Average metrics
avg_metrics = pd.DataFrame(metrics_list).mean()
print("\n📊 Average Cross-Validation Metrics:")
for k, v in avg_metrics.items():
    print(f"✅ {k.capitalize()}: {v:.4f}")